# Tutorial 8 — Temperature Ramp Analysis

Extract $T_g$ and interpolated viscoelastic properties from a
temperature-ramp DMA sweep using
[`TemperatureRampAnalysis`](../src/hermes_rheo/transforms/temperature_ramp_analysis.py).
The glass-transition temperature is located at the peak of $\tan(\delta)$
by prominence-based peak finding.

:::{note}
This tutorial demonstrates both a single-file workflow and a batch
workflow over every `.txt` in a directory.
:::

In [1]:
import os
from pathlib import Path

from hermes_rheo import read_file
from hermes_rheo.file_readers.trios_rheo_txt import TriosRheoReader
from hermes_rheo.transforms.temperature_ramp_analysis import TemperatureRampAnalysis

from hermes_rheo.visualization import use_science_style
use_science_style()

current_dir = Path(os.path.dirname(os.path.abspath("__file__")))
DATA_DIR = current_dir / "tutorial_data" / "tutorial_8"
FILE = DATA_DIR / "temp_ramp.txt"

In [2]:
experiment = read_file(
    str(FILE),
    file_reader=TriosRheoReader,
    create_composite_datasets=True,
)

print("Steps in file:")
for m in experiment.measurements:
    print(f"  {m.conditions.get('method', '<none>')!r}")

pipeline = TemperatureRampAnalysis(
    steps_to_merge=["Temperature Ramp"],   # case-insensitive substring match
    use_temps=[-20.0, 0.0, 25.0, 60.0, 100.0, 150.0],
)
result_experiment = pipeline(experiment)

print(f"\nTg = {result_experiment.tg:.2f} °C")

Steps in file:
  'Temperature ramp - 1'
  'Temperature ramp - 2'
  'Temperature ramp - 3'

Tg = -11.89 °C


In [3]:
ds = result_experiment.dataset
print("Variables:", ds.data_array_names)
print("Units:    ", ds.data_array_units)
print(f"Points:    {len(ds.data_arrays[0])}")

print("\n--- Extracted at use_temps ---")
ext = result_experiment.extracted_dataset
header = "  ".join(f"{n:>18}" for n in ext.data_array_names)
print(header)
for row in zip(*ext.data_arrays):
    print("  ".join(f"{v:>18.4g}" for v in row))

pipeline.result.to_excel("/tmp/temperature_ramp_result.xlsx")
print("\nExcel → /tmp/temperature_ramp_result.xlsx")

Variables: ['angular frequency', 'step time', 'temperature', 'axial force', 'gap', 'raw phase', 'torque', 'velocity', 'oscillation torque', 'oscillation displacement', 'run time', 'time', 'stress', 'stress (step)', 'oscillation stress', 'oscillation torque (sample)', 'oscillation strain', 'oscillation stress (cmd)', 'loss modulus', 'storage modulus', 'normal stress', 'shear rate', 'oscillation strain (cmd)', 'strain constant', 'viscosity', 'torque (step)', 'date and time', '1/temperature', 'normal stress coefficient', 'phase angle', 'frequency', 'tan(delta)', 'complex modulus', 'dynamic viscosity', 'out of phase component of η*', 'complex viscosity', 'complex compliance', 'storage compliance', 'loss compliance', 'G*/sin(delta)', 'G*.sin(delta)', 'stiffness', 'oscillation strain rate', 'temperature']
Units:     ['rad/s', 's', '°C', 'N', 'µm', '°', 'µN.m', 'rad/s', 'µN.m', 'rad', 's', 's', 'Pa', 'Pa', 'Pa', 'µN.m', '%', 'Pa', 'Pa', 'Pa', 'Pa', '1/s', '%', '1/rad', 'cP', 'µN.m', '', '1/K'


Excel → /tmp/temperature_ramp_result.xlsx


In [4]:
batch_pipeline = TemperatureRampAnalysis(
    steps_to_merge=["Temperature Ramp"],
    use_temps=[-20.0, 0.0, 25.0, 60.0, 100.0, 150.0],
)


batch_results = []
for path in sorted(DATA_DIR.glob("*.txt")):
    try:
        exp = read_file(
            str(path),
            file_reader=TriosRheoReader,
            create_composite_datasets=True,
        )
        result_exp = batch_pipeline(exp)
        batch_results.append({"file": path.name, "tg": result_exp.tg})
    except Exception as exc:
        batch_results.append({"file": path.name, "tg": None})
        print(f"[WARN] {path.name}: {exc}")

print(f"\n{'File':<40} {'Tg (°C)':>10}")
print("-" * 52)
for r in batch_results:
    tg_str = f"{r['tg']:.2f}" if r["tg"] is not None else "N/A"
    print(f"{r['file']:<40} {tg_str:>10}")


File                                        Tg (°C)
----------------------------------------------------
temp_ramp.txt                                -11.89
temp_ramp2.txt                               -32.54
temp_ramp3.txt                               -34.06
temp_ramp4.txt                               -32.32
